In [12]:
# Neural Identifier Training - 2-DOF Planar Manipulator
# Methods: EKF, UKF, UKPF (Unscented Kalman Particle Filter)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time


In [14]:


# ============================================================
# 1) True nonlinear system (2-DOF Robot Arm)
# ============================================================
def plant_dynamics(state, u):
    m1, m2, l1, l2, g = 1.0, 1.0, 1.0, 1.0, 9.81
    q1, q2, dq1, dq2 = state
    tau1, tau2 = u

    # --- Mass Matrix M(q) ---
    c2 = np.cos(q2)
    s2 = np.sin(q2)
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * c2
    M12 = m2 * l2**2 + m2 * l1 * l2 * c2
    M21 = M12
    M22 = m2 * l2**2
    M = np.array([[M11, M12], [M21, M22]])

    # --- Coriolis Matrix C(q, dq) ---
    h = -m2 * l1 * l2 * s2
    C = np.array([[h * dq2, h * (dq1 + dq2)], [-h * dq1, 0.0]])

    # --- Gravity Vector G(q) ---
    G1 = (m1 + m2) * g * l1 * np.sin(q1) + m2 * g * l2 * np.sin(q1 + q2)
    G2 = m2 * g * l2 * np.sin(q1 + q2)
    G = np.array([G1, G2])

    damping = 0.5 * np.array([dq1, dq2])
    rhs = np.array([tau1, tau2]) - (C @ np.array([dq1, dq2])) - G - damping
    ddq = np.linalg.solve(M, rhs)
    
    return np.concatenate(([dq1, dq2], ddq))

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-3):
    x_dot = plant_dynamics(x_k, u_k)
    noise = np.random.standard_cauchy(4) * process_noise_std
    return x_k + dt * x_dot + noise

def add_measurement_noise(x_true, k):
    """
    Add realistic sensor noise for robotic manipulator:
    - Position sensors (encoders): Gaussian + quantization + rare outliers
    - Velocity sensors: Gaussian + impulsive spikes (heavy-tailed)
    """
    y_meas = x_true.copy()
    
    # Angle measurements (q1, q2) - Encoder noise
    # Gaussian noise + quantization (encoder resolution ~0.001 rad)
    encoder_noise = np.random.randn(2) * 0.005  # 5 mrad std
    quantization = np.round(y_meas[:2] / 0.001) * 0.001 - y_meas[:2]
    y_meas[:2] += encoder_noise + quantization
    
    # Rare electromagnetic interference (1% chance of spike)
    if np.random.rand() < 0.01:
        spike_idx = np.random.randint(0, 2)
        y_meas[spike_idx] += np.random.randn() * 0.05  # Large spike
    
    # Velocity measurements (dq1, dq2) - Tachometer/derivative noise
    # Mixed Gaussian + impulsive noise (heavier tails)
    vel_gaussian = np.random.randn(2) * 0.02  # Base noise
    
    # Impulsive component (10% chance of spike per velocity)
    for i in range(2):
        if np.random.rand() < 0.10:
            vel_gaussian[i] += np.random.randn() * 0.1  # Impulsive spike
    
    y_meas[2:] += vel_gaussian
    
    # Small bias drift (very slow, sinusoidal)
    bias_drift = 0.001 * np.sin(k * 0.01)
    y_meas += bias_drift
    
    return y_meas

# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=0.5):
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input):
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    s_q1, s_q2 = sigmoidal(q1), sigmoidal(q2)
    s_dq1, s_dq2 = sigmoidal(dq1), sigmoidal(dq2)
    
    return np.array([
        s_q2 * s_dq1,
        s_q2 * s_dq2,
        s_dq1**2,
        s_dq2**2,
        # tau1 * 0.1,
        # tau2 * 0.1,
        # 1.0
    ])

# ============================================================
# 3) Trainers (EKF, UKF, UKPF)
# ============================================================
class Generic_RHONN_Trainer:
    def get_prediction(self, weights, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=1.0, P0=1.0, Q=1e-2, R=1e-3):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights)*P0 for _ in range(n_neurons)]
        self.Q = np.eye(n_weights)*Q
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        H = z.reshape(-1, 1)
        for i in range(self.n_neurons):
            S = self.R + (H.T @ self.P[i] @ H)[0,0]
            K = (self.P[i] @ H).flatten() / S
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            self.weights[i] += self.eta * K * err
            self.P[i] = self.P[i] - np.outer(K, H.flatten()) @ self.P[i] + self.Q

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=1.5, alpha=1e-2):
        self.n_neurons = n_neurons
        self.n = n_weights
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights) for _ in range(n_neurons)]
        self.Q = np.eye(n_weights)*1e-4
        self.R = 1e-3
        self.eta = eta
        
        self.lambd = alpha**2 * self.n - self.n
        self.Wm = np.full(2*self.n+1, 1/(2*(self.n+self.lambd)))
        self.Wc = np.copy(self.Wm)
        self.Wm[0] = self.lambd/(self.n+self.lambd)
        self.Wc[0] = self.Wm[0] + (3 - alpha**2)

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        for i in range(self.n_neurons):
            try: L = np.linalg.cholesky((self.n + self.lambd) * self.P[i])
            except: L = np.eye(self.n) * 0.1
            
            sigmas = np.zeros((2*self.n+1, self.n))
            sigmas[0] = self.weights[i]
            for k in range(self.n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[self.n+k+1] = self.weights[i] - L[:,k]
            
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(self.Wm * Y_sigmas)
            Py = np.sum(self.Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(self.n)
            for k in range(2*self.n+1):
                Pxy += self.Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
            
            K = Pxy / Py
            self.weights[i] += self.eta * K * (x_kp1[i] - y_mean)
            self.P[i] = self.P[i] - np.outer(K, K) * Py + self.Q

class UKPF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, n_particles=30, alpha=1e-3, beta=2):
        self.n_neurons = n_neurons
        self.n_weights = n_weights
        self.n_particles = n_particles
        self.particles = [np.random.randn(n_particles, n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [[np.eye(n_weights)*0.5 for _ in range(n_particles)] for _ in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        
        self.n = n_weights
        self.lambd = alpha**2 * self.n - self.n
        self.Wm = np.full(2*self.n+1, 1/(2*(self.n+self.lambd)))
        self.Wc = np.copy(self.Wm)
        self.Wm[0] = self.lambd/(self.n+self.lambd)
        self.Wc[0] = self.Wm[0] + (1 - alpha**2 + beta)
        
        self.R_local = 1e-2
        self.Q_local = 1e-3

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        for i in range(self.n_neurons):
            for p in range(self.n_particles):
                try: L = np.linalg.cholesky((self.n + self.lambd) * self.P[i][p])
                except: L = np.eye(self.n) * 1e-2
                
                sigs = np.zeros((2*self.n+1, self.n))
                sigs[0] = self.particles[i][p]
                for j in range(self.n):
                    sigs[j+1] = self.particles[i][p] + L[:,j]
                    sigs[self.n+j+1] = self.particles[i][p] - L[:,j]
                
                Y_sigs = sigs @ z
                y_pred = np.sum(self.Wm * Y_sigs)
                Py = np.sum(self.Wc * (Y_sigs - y_pred)**2) + self.R_local
                Pxy = np.zeros(self.n)
                for j in range(2*self.n+1):
                    Pxy += self.Wc[j] * (sigs[j] - self.particles[i][p]) * (Y_sigs[j] - y_pred)
                
                K = Pxy / Py
                innov = x_kp1[i] - y_pred
                self.particles[i][p] += K * innov
                self.P[i][p] -= np.outer(K, K) * Py + self.Q_local
                
                self.weights_pf[i][p] *= (np.exp(-0.5 * (innov**2 / self.R_local)) + 1e-300)
            
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            if 1.0/np.sum(self.weights_pf[i]**2) < self.n_particles/2:
                indices = np.searchsorted(np.cumsum(self.weights_pf[i]), (np.arange(self.n_particles) + np.random.uniform())/self.n_particles)
                self.particles[i] = self.particles[i][indices]
                self.P[i] = [self.P[i][idx].copy() for idx in indices]
                self.weights_pf[i].fill(1.0/self.n_particles)

    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) for i in range(self.n_neurons)]

# ============================================================
# 4) Simulation Main Loop
# ============================================================
if __name__ == "__main__":
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    n_states, n_features = 4, 4
    
    ekf = EKF_Trainer(n_states, n_features)
    ukf = UKF_Trainer(n_states, n_features)
    ukpf = UKPF_Trainer(n_states, n_features, n_particles=8)
    
    x_true = np.zeros((n_steps, 4))
    x_meas = np.zeros((n_steps, 4))  # Noisy measurements
    x_est_ekf, x_est_ukf, x_est_ukpf = [np.zeros((n_steps, 4)) for _ in range(3)]
    
    x_true[0] = [-np.pi/2, 0.2, 0, 0]
    x_meas[0] = add_measurement_noise(x_true[0], 0)
    u_hist = np.zeros((n_steps, 2))

    # Arrays para medir tiempos
    times = {'ekf': [], 'ukf': [], 'ukpf': []}

    print("Simulando Identificación con UKPF...")
    print("Ruido de medición: Encoders + cuantización + outliers (ángulos)")
    print("                   Tacómetros + ruido impulsivo (velocidades)")
    print()
    
    for k in range(n_steps - 1):
        u_hist[k] = [30.0 * np.sin(2.5 * t[k]), 15.0 * np.cos(3.5 * t[k])]
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        x_meas[k+1] = add_measurement_noise(x_true[k+1], k+1)
        
        # Updates con medición de tiempo (using NOISY measurements)
        t0 = time.perf_counter()
        ekf.update(x_meas[k+1], x_meas[k], u_hist[k])
        times['ekf'].append(time.perf_counter() - t0)
        
        t0 = time.perf_counter()
        ukf.update(x_meas[k+1], x_meas[k], u_hist[k])
        times['ukf'].append(time.perf_counter() - t0)
        
        t0 = time.perf_counter()
        ukpf.update(x_meas[k+1], x_meas[k], u_hist[k])
        times['ukpf'].append(time.perf_counter() - t0)
        
        # Predictions
        z = construct_z_vector(x_meas[k], u_hist[k])
        w_ukpf = ukpf.get_estimates()
        for i in range(4):
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            x_est_ukpf[k+1, i] = np.dot(w_ukpf[i], z)
        
        if k % 50 == 0: print(f"Paso {k}")

    # --- Métricas detalladas ---
    def compute_metrics(x_true, x_est, name):
        errors = x_true - x_est
        mse = np.mean(errors**2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(errors))
        
        state_names = ['q1', 'q2', 'dq1', 'dq2']
        print(f"\n{name}:")
        print(f"  MSE Global:  {mse:.6f}")
        print(f"  RMSE Global: {rmse:.6f}")
        print(f"  MAE Global:  {mae:.6f}")
        print(f"  RMSE por estado:")
        for i, sname in enumerate(state_names):
            rmse_i = np.sqrt(np.mean(errors[:, i]**2))
            print(f"    {sname}: {rmse_i:.6f}")
        
        return mse, rmse, mae

    print("\n" + "="*60)
    print("RESULTADOS FINALES")
    print("="*60)
    
    mse_ekf, rmse_ekf, mae_ekf = compute_metrics(x_true, x_est_ekf, "EKF")
    mse_ukf, rmse_ukf, mae_ukf = compute_metrics(x_true, x_est_ukf, "UKF")
    mse_ukpf, rmse_ukpf, mae_ukpf = compute_metrics(x_true, x_est_ukpf, "UKPF")

    # Tiempos de cómputo
    print("\n" + "="*60)
    print("TIEMPOS DE CÓMPUTO (promedio por paso)")
    print("="*60)
    for method in ['ekf', 'ukf', 'ukpf']:
        avg_time = np.mean(times[method]) * 1000
        print(f"{method.upper()}: {avg_time:.4f} ms")

    # Gráfica de Estados
    fig = make_subplots(rows=2, cols=2, 
                        subplot_titles=("q₁ (Articulación 1)", "q₂ (Articulación 2)", 
                                       "dq₁ (Velocidad 1)", "dq₂ (Velocidad 2)"))
    
    colors = {'Real': 'black', 'Medido': 'gray', 'EKF': 'blue', 'UKF': 'green', 'UKPF': 'red'}
    dashes = {'Real': 'solid', 'Medido': 'dot', 'EKF': 'dash', 'UKF': 'dot', 'UKPF': 'dashdot'}
    
    data_dict = {'Real': x_true, 'Medido': x_meas, 'EKF': x_est_ekf, 'UKF': x_est_ukf, 'UKPF': x_est_ukpf}
    
    for idx in range(4):
        row, col = (idx // 2) + 1, (idx % 2) + 1
        for name, data in data_dict.items():
            fig.add_trace(go.Scatter(
                x=t, y=data[:, idx], 
                name=name if idx == 0 else None,
                legendgroup=name,
                showlegend=(idx == 0),
                line=dict(color=colors[name], dash=dashes[name], 
                         width=2.5 if name == 'Real' else (1.0 if name == 'Medido' else 2)),
                opacity=0.6 if name == 'Medido' else 1.0
            ), row=row, col=col)
    
    fig.update_layout(
        height=700, 
        width=1200,
        template="plotly_white", 
        title="Identificación RHONN con Ruido de Sensores: EKF vs UKF vs UKPF",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )
    fig.show()

    # Gráfica de Errores
    fig2 = make_subplots(rows=2, cols=2, 
                         subplot_titles=("Error q₁", "Error q₂", "Error dq₁", "Error dq₂"))
    
    errors_dict = {
        'Medido': x_true - x_meas,
        'EKF': x_true - x_est_ekf,
        'UKF': x_true - x_est_ukf,
        'UKPF': x_true - x_est_ukpf
    }
    
    for idx in range(4):
        row, col = (idx // 2) + 1, (idx % 2) + 1
        for name, err in errors_dict.items():
            fig2.add_trace(go.Scatter(
                x=t, y=err[:, idx],
                name=name if idx == 0 else None,
                legendgroup=name,
                showlegend=(idx == 0),
                line=dict(color=colors[name])
            ), row=row, col=col)
        
        fig2.add_hline(y=0, line_dash="dash", line_color="gray", 
                      opacity=0.5, row=row, col=col)
    
    fig2.update_layout(
        height=700,
        width=1200,
        template="plotly_white",
        title="Análisis de Errores de Identificación",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )
    fig2.show()

    print("\n✓ Simulación completada")

Simulando Identificación con UKPF...
Ruido de medición: Encoders + cuantización + outliers (ángulos)
                   Tacómetros + ruido impulsivo (velocidades)

Paso 0
Paso 50
Paso 100
Paso 150
Paso 200
Paso 250
Paso 300
Paso 350
Paso 400
Paso 450
Paso 500
Paso 550
Paso 600
Paso 650
Paso 700
Paso 750
Paso 800
Paso 850
Paso 900
Paso 950

RESULTADOS FINALES

EKF:
  MSE Global:  0.131285
  RMSE Global: 0.362332
  MAE Global:  0.057385
  RMSE por estado:
    q1: 0.219741
    q2: 0.636958
    dq1: 0.178223
    dq2: 0.198429

UKF:
  MSE Global:  1.089218
  RMSE Global: 1.043656
  MAE Global:  0.345638
  RMSE por estado:
    q1: 1.184748
    q2: 1.456909
    dq1: 0.547202
    dq2: 0.728857

UKPF:
  MSE Global:  0.005252
  RMSE Global: 0.072469
  MAE Global:  0.017396
  RMSE por estado:
    q1: 0.059044
    q2: 0.112918
    dq1: 0.040603
    dq2: 0.055874

TIEMPOS DE CÓMPUTO (promedio por paso)
EKF: 0.0426 ms
UKF: 0.1202 ms
UKPF: 0.8235 ms



✓ Simulación completada


In [11]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1) Plant Dynamics (2-DOF Robotic Arm)
# ============================================================
def plant_dynamics(x, u):
    """
    Simplified dynamics for a 2-DOF planar manipulator.
    x = [q1, q2, dq1, dq2]
    """
    q1, q2, dq1, dq2 = x
    tau1, tau2 = u
    
    # Inertia and Coriolis proxies
    # These represent the physical structure the RHONN must learn
    ddq1 = 0.5 * tau1 - 0.2 * dq1 - 0.1 * np.cos(q1)
    ddq2 = 0.8 * tau2 - 0.3 * dq2 - 0.1 * np.cos(q2 + q1)
    
    return np.array([dq1, dq2, ddq1, ddq2])

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-2):
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Heavy-tailed noise (Cauchy) to challenge the filters
    noise = np.random.standard_cauchy(4) * process_noise_std
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=0.5):
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input):
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    s_q1, s_q2 = sigmoidal(q1), sigmoidal(q2)
    s_dq1, s_dq2 = sigmoidal(dq1), sigmoidal(dq2)
    
    return np.array([
        # s_q1, s_q2, s_dq1, s_dq2,     
        s_q2 * s_dq1,                 
        s_q2 * s_dq2,                 
        s_dq1**2, s_dq2**2            
    ])

# ============================================================
# 3) Trainers (EKF, UKF, PF, UKPF)
# ============================================================
class Generic_RHONN_Trainer:
    def get_prediction(self, weights, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=1.0, P0=1.0, Q=1e-2, R=1e-3):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights)*P0 for _ in range(n_neurons)]
        self.Q = np.eye(n_weights)*Q
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        H = z.reshape(-1, 1)
        for i in range(self.n_neurons):
            S = self.R + (H.T @ self.P[i] @ H)[0,0]
            K = (self.P[i] @ H).flatten() / S
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            self.weights[i] += self.eta * K * err
            self.P[i] = self.P[i] - np.outer(K, H.flatten()) @ self.P[i] + self.Q

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, eta=2.0, alpha=8e-3):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [np.eye(n_weights) for _ in range(n_neurons)]
        self.Q, self.R = np.eye(n_weights)*3e-4, 1e-3
        self.eta, self.n = eta, n_weights
        self.lambd = alpha**2 * self.n - self.n
        self.Wm = np.full(2*self.n+1, 1/(2*(self.n+self.lambd)))
        self.Wc = np.copy(self.Wm)
        self.Wm[0] = self.lambd/(self.n+self.lambd)
        self.Wc[0] = self.Wm[0] + (3 - alpha**2)

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        for i in range(self.n_neurons):
            try: L = np.linalg.cholesky((self.n + self.lambd) * self.P[i])
            except: L = np.eye(self.n) * 0.1
            sigs = np.zeros((2*self.n+1, self.n))
            sigs[0] = self.weights[i]
            for k in range(self.n):
                sigs[k+1], sigs[self.n+k+1] = self.weights[i] + L[:,k], self.weights[i] - L[:,k]
            Y_sigs = sigs @ z
            y_mean = np.sum(self.Wm * Y_sigs)
            Py = np.sum(self.Wc * (Y_sigs - y_mean)**2) + self.R
            Pxy = np.sum([self.Wc[k]*np.outer(sigs[k]-self.weights[i], Y_sigs[k]-y_mean) for k in range(2*self.n+1)], axis=0).flatten()
            K = Pxy / Py
            self.weights[i] += self.eta * K * (x_kp1[i] - y_mean)
            self.P[i] = (self.P[i] - np.outer(K, K) * Py) + np.eye(self.n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, n_particles=200, Q_std=0.6, R_std=0.2):
        self.n_neurons, self.n_weights, self.n_particles = n_neurons, n_weights, n_particles
        self.particles = [np.random.randn(n_particles, n_weights)*0.2 for _ in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.Q_std = np.array(Q_std) if not np.isscalar(Q_std) else np.ones(n_neurons)*Q_std
        self.R_std = np.array(R_std) if not np.isscalar(R_std) else np.ones(n_neurons)*R_std

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        for i in range(self.n_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.n_weights) * self.Q_std[i]
            err = x_kp1[i] - (self.particles[i] @ z)
            self.weights_pf[i] *= (np.exp(-0.5 * (err/self.R_std[i])**2) + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            if 1.0 / np.sum(self.weights_pf[i]**2) < self.n_particles/2:
                idx = np.searchsorted(np.cumsum(self.weights_pf[i]), (np.arange(self.n_particles) + np.random.uniform())/self.n_particles)
                self.particles[i], self.weights_pf[i] = self.particles[i][idx], np.ones(self.n_particles)/self.n_particles

    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) for i in range(self.n_neurons)]

class UKPF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_weights, n_particles=30, alpha=1e-3, beta=2):
        self.n_neurons, self.n_weights, self.n_particles = n_neurons, n_weights, n_particles
        self.particles = [np.random.randn(n_particles, n_weights)*0.1 for _ in range(n_neurons)]
        self.P = [[np.eye(n_weights)*0.5 for _ in range(n_particles)] for _ in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.n = n_weights
        self.lambd = alpha**2 * self.n - self.n
        self.Wm = np.full(2*self.n+1, 1/(2*(self.n+self.lambd)))
        self.Wc = np.copy(self.Wm)
        self.Wm[0], self.Wc[0] = self.lambd/(self.n+self.lambd), self.lambd/(self.n+self.lambd) + (1 - alpha**2 + beta)
        self.R_local, self.Q_local = 1e-2, 1e-3

    def update(self, x_kp1, x_k, u_k):
        z = construct_z_vector(x_k, u_k)
        for i in range(self.n_neurons):
            for p in range(self.n_particles):
                try: L = np.linalg.cholesky((self.n + self.lambd) * self.P[i][p])
                except: L = np.eye(self.n) * 1e-2
                sigs = np.zeros((2*self.n+1, self.n))
                sigs[0] = self.particles[i][p]
                for j in range(self.n):
                    sigs[j+1], sigs[self.n+j+1] = self.particles[i][p] + L[:,j], self.particles[i][p] - L[:,j]
                Y_sigs = sigs @ z
                y_pred = np.sum(self.Wm * Y_sigs)
                Py = np.sum(self.Wc * (Y_sigs - y_pred)**2) + self.R_local
                Pxy = np.sum([self.Wc[j]*np.outer(sigs[j]-self.particles[i][p], Y_sigs[j]-y_pred) for j in range(2*self.n+1)], axis=0).flatten()
                K, innov = Pxy / Py, x_kp1[i] - y_pred
                self.particles[i][p] += K * innov
                self.P[i][p] -= np.outer(K, K) * Py + self.Q_local
                self.weights_pf[i][p] *= (np.exp(-0.5 * (innov**2 / self.R_local)) + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            if 1.0/np.sum(self.weights_pf[i]**2) < self.n_particles/2:
                idx = np.searchsorted(np.cumsum(self.weights_pf[i]), (np.arange(self.n_particles) + np.random.uniform())/self.n_particles)
                self.particles[i] = self.particles[i][idx]
                self.P[i] = [self.P[i][k].copy() for k in idx]
                self.weights_pf[i].fill(1.0/self.n_particles)

    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) for i in range(self.n_neurons)]

# ============================================================
# 4) Main Simulation
# ============================================================
if __name__ == "__main__":
    n_steps, dt = 500, 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    n_states, n_features = 4, 4
    measurement_noise = np.random.laplace(0, 0.01, (n_steps, 4))

    ekf = EKF_Trainer(n_states, n_features)
    ukf = UKF_Trainer(n_states, n_features)
    pf = PF_Trainer(n_states, n_features, n_particles=800,Q_std=[0.5, 0.5, 0.5, 0.5], R_std=[0.01, 0.01, 0.1, 0.1])
    ukpf = UKPF_Trainer(n_states, n_features, n_particles=30)

    x_true = np.zeros((n_steps, 4)) + measurement_noise
    x_est_ekf, x_est_ukf, x_est_pf, x_est_ukpf = [np.zeros((n_steps, 4)) for _ in range(4)]
    
    x_true[0] = [-np.pi/2, 0, 0, 0]
    for arr in [x_est_ekf, x_est_ukf, x_est_pf, x_est_ukpf]: arr[0] = x_true[0]

    u_hist = np.column_stack([30*np.sin(2*t), 15*np.cos(3*t)])

    print("Simulating 2-DOF Arm Identification...")
    for k in range(n_steps - 1):
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        z = construct_z_vector(x_true[k], u_hist[k])
        
        ekf.update(x_true[k+1], x_true[k], u_hist[k])
        ukf.update(x_true[k+1], x_true[k], u_hist[k])
        pf.update(x_true[k+1], x_true[k], u_hist[k])
        ukpf.update(x_true[k+1], x_true[k], u_hist[k])

        w_pf, w_ukpf = pf.get_estimates(), ukpf.get_estimates()
        for i in range(4):
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z)
            x_est_ukpf[k+1, i] = np.dot(w_ukpf[i], z)

    # ============================================================
    # 5) Thesis Visualization
    # ============================================================
    print("\nCalculating MSE and Generating Plots...")
    config = {'font': 'Computer Modern, serif', 'lw_t': 3, 'lw_e': 2}
    
    for i, label in enumerate(['q₁ (rad)', 'q₂ (rad)', 'dq₁ (rad/s)', 'dq₂ (rad/s)']):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=t, y=x_true[:,i], name='Real', line=dict(color='black', width=config['lw_t'])))
        fig.add_trace(go.Scatter(x=t, y=x_est_ekf[:,i], name='EKF', line=dict(dash='dash')))
        fig.add_trace(go.Scatter(x=t, y=x_est_ukf[:,i], name='UKF', line=dict(dash='dot')))
        fig.add_trace(go.Scatter(x=t, y=x_est_pf[:,i], name='PF', line=dict(dash='dashdot')))
        fig.add_trace(go.Scatter(x=t, y=x_est_ukpf[:,i], name='UKPF', line=dict(color='orange', width=config['lw_e'])))
        
        fig.update_layout(title=f"Tracking Performance: {label}", xaxis_title="Time (s)", yaxis_title=label, template="plotly_white")
        fig.show()

    # Bar Chart for MSE Comparison
    filters = ['EKF', 'UKF', 'PF', 'UKPF']
    total_mse = [np.mean((x_true - x)**2) for x in [x_est_ekf, x_est_ukf, x_est_pf, x_est_ukpf]]
    
    fig_mse = go.Figure([go.Bar(x=filters, y=total_mse, marker_color=['blue', 'green', 'red', 'orange'])])
    fig_mse.update_layout(title="Total Mean Squared Error (Log Scale)", yaxis_type="log", template="plotly_white")
    fig_mse.show()

    print("✅ Full Simulation and Analysis complete.")

Simulating 2-DOF Arm Identification...

Calculating MSE and Generating Plots...


✅ Full Simulation and Analysis complete.
